# r/Cornell Synthetic Subreddit Simulation Notebook

> **Note:** This notebook is designed to be used in conjunction with the
> [AgentSociety GitHub repository](https://github.com/tsinghua-fib-lab/agentsociety).
> It simply wraps a few helper scripts into a documented, end-to-end pipeline.

The goal of this notebook is to:

- Extract **topics** (thread titles) from the real r/Cornell ConvoKit corpus.
- Compute basic **per-topic statistics** (unique users and number of comments).
- Generate a large set of **synthetic user profiles** for r/Cornell-style users.
- Use **AgentSociety + vLLM** to simulate subreddit-style conversations on those topics.


## Overview of the Pipeline

The pipeline is split into four main steps, each corresponding to one of the original scripts:

1. **Build topic list from the ConvoKit r/Cornell corpus**  
   - Script: `make_cornell_topics_json.py`  
   - Output: `cornell_topics.json` containing conversation IDs and their titles.

2. **Compute corpus statistics per topic**  
   - Script: `cornell_stats.py`  
   - Prints statistics such as average number of unique users and comments per topic.
   - These values are later hard-coded into the subreddit simulation script.

3. **Generate synthetic user profiles**  
   - Script: `generate_profiles.py`  
   - Output: `profiles_cornell.json` with 10,000 simulated Cornell subreddit users.

4. **Generate synthetic subreddit-style conversations with AgentSociety**  
   - Script: `generate_cornell_subreddit.py`  
   - Uses `profiles_cornell.json`, `cornell_topics_all.json`, and a running vLLM server
     to sample conversations and save them in a Convokit-style JSON file.


## Prerequisites

Before running this notebook end-to-end, make sure you have:

- **Python 3.8+** (or compatible version used by AgentSociety).
- The [`convokit`](https://convokit.cornell.edu/) library installed for working with the r/Cornell corpus.
- The **AgentSociety** codebase cloned locally (see the
  [AgentSociety GitHub repository](https://github.com/tsinghua-fib-lab/agentsociety)).
- A **vLLM** server running locally (or a compatible OpenAI-style endpoint) for text generation.

Typical Python dependencies (non-exhaustive):

- `convokit`
- `requests`
- Any dependencies required by AgentSociety (LLM wrapper, config classes, etc.).

You can install some of these with e.g.:

```bash
pip install convokit requests
```

For AgentSociety itself, follow the installation instructions in the GitHub repository.


### (Optional) Quick Dependency Check

You can use the cell below to quickly check that core dependencies import correctly.
If any import fails, install the missing package before proceeding.


In [ ]:
# Quick import check (run this cell to verify imports)
# If any of these fail, install the missing package (e.g., via pip).

try:
    import convokit
    print("convokit imported successfully.")
except ImportError:
    print("convokit is not installed. Install it with `pip install convokit`.")

try:
    import requests
    print("requests imported successfully.")
except ImportError:
    print("requests is not installed. Install it with `pip install requests`.")

# AgentSociety imports will depend on where you cloned the repository and your PYTHONPATH.
try:
    from agentsociety.llm.llm import LLM, LLMConfig, LLMProviderType
    print("AgentSociety imports seem to work.")
except Exception as e:
    print("AgentSociety is not importable yet. Make sure the repo is installed / on PYTHONPATH.")
    print("Error:", e)


## Step 1 – Build Topic List from the r/Cornell ConvoKit Corpus

This step:

- Downloads (if necessary) and loads the **r/Cornell** subreddit corpus from ConvoKit.
- Iterates over all conversations (threads) in the corpus.
- Extracts the **title** for each conversation and builds a dictionary:
  ```python
  { "<conversation_id>": "Thread title", ... }
  ```
- Saves the dictionary to a JSON file (by default `cornell_topics.json`) with structure:

```json
{
  "TOPICS": {
    "<conversation_id_1>": "title 1",
    "<conversation_id_2>": "title 2"
  }
}
```

You can adapt the output filename if you need a different name such as `cornell_topics_all.json`
to match other scripts.


In [ ]:
import json
from typing import Dict

from convokit import Corpus, download  # pip install convokit


SUBREDDIT_NAME = "Cornell"
CORPUS_NAME = f"subreddit-{SUBREDDIT_NAME}"


def load_cornell_corpus() -> Corpus:
    """Download (if needed) and load the r/Cornell subreddit Corpus."""
    print(f"Downloading/loading corpus '{CORPUS_NAME}'...")
    corpus = Corpus(filename=download(CORPUS_NAME))
    print("Loaded corpus.")
    return corpus


def build_topics_dict_from_titles(corpus: Corpus) -> Dict[str, str]:
    """
    Build a TOPICS dict mapping conversation_id -> title for **all** topics.

    Structure:
        {
            "<conversation_id>": "Some title",
            "<conversation_id_2>": "Another title",
            ...
        }

    Notes:
    - conversation_id is taken from convo.id (stringified for JSON keys).
    - Only conversations with a non-empty title are included.
    """
    topics: Dict[str, str] = {}

    for convo in corpus.iter_conversations():
        title = convo.meta.get("title")
        if title and isinstance(title, str):
            title = title.strip()
            if title:
                # convo.id is the conversation_id; stringify it for JSON
                topics[str(convo.id)] = title

    if not topics:
        raise ValueError("No conversation titles found in the corpus.")

    print(f"Collected {len(topics)} topics with titles.")
    return topics


def save_topics_json(
    topics: Dict[str, str],
    out_path: str = "cornell_topics.json"
) -> None:
    """
    Save topics into a JSON file with the structure:

    {
      "TOPICS": {
        "<conversation_id_1>": "title 1",
        "<conversation_id_2>": "title 2",
        ...
      }
    }
    """
    data = {"TOPICS": topics}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"Saved topics JSON to: {out_path}")


def main():
    corpus = load_cornell_corpus()
    topics = build_topics_dict_from_titles(corpus)
    save_topics_json(topics, out_path="cornell_topics.json")

    # Optional: print a Python-style representation for your reference
    print("\nPython-style representation (first 20 entries):\n")
    print("TOPICS: Dict[str, str] = {")
    for i, (cid, title) in enumerate(topics.items()):
        if i >= 20:
            print("    # ...")
            break
        safe_title = title.replace('"', '\\"')
        print(f'    "{cid}": "{safe_title}",')
    print("}")


if __name__ == "__main__":
    main()


## Step 2 – Compute Per-Topic Corpus Statistics

This step computes statistics over the **original** r/Cornell corpus:

- For each topic (conversation):
  - The number of **unique users** participating.
  - The number of **comments** (excluding the root post).
- It then prints:
  - Number of topics.
  - Min, max, and average unique users per topic.
  - Min, max, and average number of comments per topic.

These summary statistics can be used to calibrate the synthetic generation process
(e.g., as done in `generate_cornell_subreddit.py` with `AVG_UNIQUE_USERS` and `AVG_COMMENTS`).


In [ ]:
"""
Compute per-topic statistics for the r/Cornell subreddit using ConvoKit.

For each conversation (topic), we compute:

- unique_users: number of UNIQUE users (speakers) who posted in the topic
- num_comments: number of comments (excluding the root post)

Then we report:

UNIQUE USERS (per topic)
- Number of topics
- Max unique users in a topic
- Average unique users per topic
- Which topic(s) have the max unique users
- Note: min unique users is logically 1 for lone-OP topics.

COMMENTS (per topic)
- Min number of comments in a topic
- Max number of comments in a topic
- Average number of comments per topic
"""

from typing import List, Tuple
import statistics

from convokit import Corpus, download  # pip install convokit

SUBREDDIT_NAME = "Cornell"
CORPUS_NAME = f"subreddit-{SUBREDDIT_NAME}"


def load_cornell_corpus() -> Corpus:
    """Download (if needed) and load the r/Cornell subreddit corpus."""
    print(f"Downloading/loading corpus '{CORPUS_NAME}'...")
    corpus = Corpus(filename=download(CORPUS_NAME))
    print("Loaded corpus.\n")
    return corpus


def collect_topic_stats(
    corpus: Corpus,
) -> List[Tuple[object, int, int]]:
    """
    For each conversation (topic), compute:

    - unique_users: number of unique user IDs participating
    - num_comments: number of comments (excluding the root post)

    Returns:
        List of (conversation, unique_users, num_comments) tuples.
    """
    results: List[Tuple[object, int, int]] = []

    for convo in corpus.iter_conversations():
        speakers = set()
        utterances = list(convo.iter_utterances())

        for utt in utterances:
            speaker = utt.speaker
            speaker_id = getattr(speaker, "id", speaker)
            speakers.add(speaker_id)

        unique_users = len(speakers)

        # Try to use Reddit's num_comments metadata if available,
        # otherwise fall back to "all utterances minus 1 root post"
        meta_num_comments = convo.meta.get("num_comments")
        if isinstance(meta_num_comments, int):
            num_comments = meta_num_comments
        else:
            # assume at least 1 utterance is the root; comments = total - 1
            num_comments = max(0, len(utterances) - 1)

        results.append((convo, unique_users, num_comments))

    return results


def print_unique_user_topic_stats(topic_stats: List[Tuple[object, int, int]]) -> None:
    """
    Given (conversation, unique_users, num_comments) tuples, compute and print:

    - number of topics
    - max unique users in a topic
    - average unique users per topic
    - which topic(s) have the max unique users
    """
    if not topic_stats:
        print("No topics found; cannot compute unique-user topic stats.")
        return

    unique_counts = [unique_users for _, unique_users, _ in topic_stats]

    max_users = max(unique_counts)
    avg_users = statistics.mean(unique_counts)

    print("=== UNIQUE USER STATISTICS PER TOPIC ===")
    print(f"Number of topics (conversations): {len(topic_stats)}")
    print(f"Max unique users in a topic     : {max_users}")
    print(f"Avg unique users per topic      : {avg_users:.2f}")
    print("Min unique users per topic      : 1 (for lone-OP topics)")
    print()

    print("Topics with MAX unique users:")
    for convo, unique_users, _ in topic_stats:
        if unique_users == max_users:
            print(f"  - Topic ID: {convo.id} | unique users: {unique_users}")
    print()


def print_comment_topic_stats(topic_stats: List[Tuple[object, int, int]]) -> None:
    """
    Given (conversation, unique_users, num_comments) tuples, compute and print:

    - min number of comments
    - max number of comments
    - average number of comments per topic
    """
    if not topic_stats:
        print("No topics found; cannot compute comment stats.")
        return

    comment_counts = [num_comments for _, _, num_comments in topic_stats]

    min_comments = min(comment_counts)
    max_comments = max(comment_counts)
    avg_comments = statistics.mean(comment_counts)

    print("=== COMMENT STATISTICS PER TOPIC ===")
    print(f"Min comments in a topic   : {min_comments}")
    print(f"Max comments in a topic   : {max_comments}")
    print(f"Avg comments per topic    : {avg_comments:.2f}")
    print()


def main():
    corpus = load_cornell_corpus()

    # Collect per-topic stats
    topic_stats = collect_topic_stats(corpus)

    # Unique user stats
    print_unique_user_topic_stats(topic_stats)

    # Comment stats
    print_comment_topic_stats(topic_stats)


if __name__ == "__main__":
    main()


## Step 3 – Generate Synthetic User Profiles

This step:

- Creates 10,000 synthetic r/Cornell users, named `cornell_user_1`, `cornell_user_2`, ...  
- For each user, it randomly assigns attributes such as:
  - `age`, `gender`, `role`, `status`, `major`, `year`
  - `personality`, `political_interest`, `subreddit_behavior`
- Saves all profiles to `profiles_cornell.json` in the form:

```json
[
  {
    "id": 1,
    "profile": { ... }
  },
  {
    "id": 2,
    "profile": { ... }
  }
]
```

These profiles are later used to give each simulated commenter a persona in the subreddit conversations.


In [ ]:
import json
from pathlib import Path
import random

OUTPUT_PATH = Path(__file__).parent / "profiles_cornell.json"

MAJORS = [
    "Computer Science", "Information Science", "Economics", "Biology",
    "Psychology", "Mathematics", "Physics", "Chemistry", "Government",
    "Sociology", "History", "Hotel Administration", "Business",
    "Mechanical Engineering", "Electrical Engineering", "Architecture",
    "Chemical Engineering", "Civil Engineering", "English", "Fine Arts",
    "Environmental Science", "Operations Research", "Statistics",
    "Urban Planning", "Linguistics", "Philosophy", "Nutritional Science",
    "Animal Science", "Industrial and Labor Relations", "Communication",
    "Agricultural Sciences", "Computer Engineering", "Human Development"
]

GENDERS = [
    "male",
    "female",
    "prefer not to say"
]

ROLES = [
    "Cornell undergraduate student and subreddit user",
    "Cornell graduate student and subreddit user",
    "Cornell PhD student and subreddit user",
    "Cornell staff member and subreddit user",
    "Cornell alumni and subreddit user",
    "Prospective Cornell student browsing the subreddit",
    # New roles
    "Cornell alumni who already graduated and subreddit user",
    "Former Cornell student who dropped out and subreddit user",
]

STATUSES = [
    "undergraduate student",
    "graduate student",
    "PhD student",
    "exchange student",
    "alumni",
    "staff",
    # New statuses
    "already graduated",
    "dropout",
]

YEARS = [
    "Freshman",
    "Sophomore",
    "Junior",
    "Senior",
    "Graduate",
    # New years
    "Already graduated",
    "Dropped out",
]

PERSONALITIES = [
    "introverted but thoughtful",
    "extroverted and humorous",
    "analytical and data-driven",
    "empathetic and supportive",
    "sarcastic but caring",
    "organized and type-A",
    "laid-back and go-with-the-flow",
    "highly ambitious and career-focused",
    "creative and artsy",
    "curious and always asking questions",
    # New more negative-leaning personalities
    "cynical and often pessimistic",
    "burnt out and disengaged",
    "defensive and easily irritated",
    "overly competitive and dismissive of others",
]

POLITICAL_INTEREST = [
    "not very interested",
    "moderately interested",
    "very interested",
    "primarily interested in campus politics",
    "active in advocacy and social issues",
]

SUBREDDIT_BEHAVIOR = [
    "mostly lurks and upvotes",
    "frequently comments and debates",
    "occasionally posts detailed threads",
    "shares memes and light-hearted content",
    "asks for course and professor advice",
    "posts about housing, leases, and roommates",
    "shares information about campus events",
    "gives advice to incoming students",
    # New more negative subreddit behaviors
    "frequently complains or rants about Cornell",
    "often posts pessimistic takes about campus issues",
    "gets into heated arguments in comment threads",
    "regularly downvotes opinions they disagree with",
]


def make_profile(user_id: int, username: str) -> dict:
    """
    A single agent profile. Structure is intentionally simple:
    AgentSociety just needs a JSON with per-agent 'profile' info.
    """
    return {
        "id": user_id,
        "profile": {
            "name": username,
            "nickname": username,
            "age": random.randint(18, 30),
            "gender": random.choice(GENDERS),
            "role": random.choice(ROLES),
            "status": random.choice(STATUSES),
            "university": "Cornell University",
            "major": random.choice(MAJORS),
            "year": random.choice(YEARS),
            "personality": random.choice(PERSONALITIES),
            "political_interest": random.choice(POLITICAL_INTEREST),
            "subreddit_behavior": random.choice(SUBREDDIT_BEHAVIOR),
            "backstory": (
                "A Cornell community member who uses the subreddit to "
                "stay informed, vent about stress, and connect with others."
            ),
        },
    }


def main():
    profiles = []
    for i in range(10000):
        username = f"cornell_user_{i+1}"
        profiles.append(make_profile(i + 1, username))

    OUTPUT_PATH.write_text(json.dumps(profiles, indent=2), encoding="utf-8")
    print(f"Wrote {len(profiles)} profiles to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


## Step 4 – Generate Synthetic Subreddit-Style Conversations with AgentSociety

This final step uses **AgentSociety + vLLM** to generate synthetic conversations
for each topic (thread title) we extracted earlier.

High-level behavior of the script:

1. **Waits for the vLLM server** to become healthy before starting.
2. **Loads profiles** from `profiles_cornell.json`.
3. **Loads topics** from `cornell_topics_all.json` (or similar topics JSON).
4. Initializes an `LLM` instance backed by vLLM (with a Qwen model in this example).
5. For each topic:
   - Samples how many **participants** will appear in the thread.
   - Samples how many **comments** the thread will have, using a truncated normal distribution
     based on statistics from the real corpus.
   - Randomly selects who speaks next in the thread, building a short **thread summary** for context.
   - Builds a **persona snippet** for the current speaker from their profile.
   - Creates a **system prompt** instructing the LLM to act as a single Reddit user on `/r/Cornell`.
   - Calls `llm.atext_request(...)` to get exactly one Reddit-style comment.
   - Links comments together via `reply_to` to form a conversation tree.

6. Stores each generated utterance in a **Convokit-style** dictionary:

```python
{
  "id": <int>,
  "speaker": "<username>",
  "conversation_id": "<topic_id>",
  "reply_to": <parent_id or None>,
  "timestamp": <monotonic integer>,
  "text": "<comment text>",
  "meta": {
    "agent_profile_id": <int>,
    "topic": "<topic title>"
  }
}
```

7. Finally, writes all messages to `cornell_subreddit_conversations_all.json`.

You can later convert this JSON into a ConvoKit `Corpus` or another data structure
for downstream analysis or simulation.


In [ ]:
import asyncio
import json
import random
from pathlib import Path
from typing import Dict, List, Any, Optional
import time
import requests

from agentsociety.llm.llm import LLM, LLMConfig, LLMProviderType


# ---------- Paths ----------

BASE_DIR = Path(__file__).resolve().parent
PROFILES_PATH = BASE_DIR / "profiles_cornell.json"
TOPICS_PATH = BASE_DIR / "cornell_topics_all.json"
OUTPUT_PATH = BASE_DIR / "cornell_subreddit_conversations_all.json"


def wait_for_vllm(base_url="http://127.0.0.1:8000", timeout=300, interval=2):
    health_url = f"{base_url}/health"
    start = time.time()
    while True:
        try:
            r = requests.get(health_url, timeout=1)
            if r.status_code == 200:
                print("vLLM is ready.")
                return
        except Exception:
            pass

        if time.time() - start > timeout:
            raise RuntimeError("Timed out waiting for vLLM to become ready")

        time.sleep(interval)


# ---------- Distribution helpers (from your stats) ----------

# From your corpus:
# - avg unique users/topic ≈ 4.92, min 1, max 53
# - avg comments/topic     ≈ 5.47, min 0, max 127

AVG_UNIQUE_USERS = 4.92
MAX_UNIQUE_USERS = 53

AVG_COMMENTS = 5.47
MAX_COMMENTS = 127


def sample_truncated_normal_int(mean: float, std: float, lo: int, hi: int) -> int:
    """
    Sample an integer from a Normal(mean, std) truncated to [lo, hi].
    Simple rejection sampling; good enough for our synthetic sim.
    """
    while True:
        x = int(round(random.gauss(mean, std)))
        if lo <= x <= hi:
            return x


# ---------- Subreddit topics ----------

def load_topics() -> Dict[str, str]:
    """
    Load topics from cornell_topics.json, returning:
      {topic_id_str: topic_title}
    """
    with open(TOPICS_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    topics_raw: Dict[str, str] = data.get("TOPICS", {})
    # Sort by topic ID as a string (works for nyx4d, o0145, etc.)
    topics_sorted: Dict[str, str] = dict(
        sorted(topics_raw.items(), key=lambda kv: kv[0])
    )
    return topics_sorted


# ---------- Helper functions ----------

def load_profiles() -> Dict[int, Dict[str, Any]]:
    """
    Load profiles_cornell.json and return {agent_id: profile_dict}.
    Expected format:

    [
      {
        "id": 1,
        "profile": {
           "name": "cornell_user_1",
           "nickname": "cornell_user_1",
           ...
        }
      },
      ...
    ]
    """
    with open(PROFILES_PATH, "r", encoding="utf-8") as f:
        raw = json.load(f)

    profiles: Dict[int, Dict[str, Any]] = {}
    for entry in raw:
        agent_id = entry["id"]
        profiles[agent_id] = entry["profile"]
    return profiles


def build_thread_summary(thread_posts: List[Dict[str, Any]]) -> str:
    """
    Turn existing posts in the thread into a short textual summary
    for the prompt. Each line looks like:

      cornell_user_12: Text of their comment...
    """
    if not thread_posts:
        return "No one has posted yet. You will start the thread."

    lines = []
    for post in thread_posts[-12:]:  # only last 10–12 to keep context small
        speaker = post["speaker"]
        text = post["text"].replace("\n", " ").strip()
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)


def build_persona_snippet(profile: Dict[str, Any]) -> str:
    """
    Extract a concise persona description from the Cornell profile.
    """
    name = profile.get("nickname") or profile.get("name")
    major = profile.get("major", "Unknown major")
    year = profile.get("year", "Unknown year")
    personality = profile.get("personality", "")
    online_habits = profile.get("online_habits", "")

    parts = [
        f"Username: {name}",
        f"Major/year: {major}, {year}",
    ]
    if personality:
        parts.append(f"Personality: {personality}")
    if online_habits:
        parts.append(f"Online habits: {online_habits}")

    return "\n".join(f"- {p}" for p in parts)


def make_system_prompt(
    topic_name: str,
    thread_summary: str,
    persona_snippet: str,
) -> str:
    """
    Create a single system prompt that tells the LLM to emit ONE Reddit-style comment
    for /r/Cornell on the given topic.
    """
    return f"""
You are simulating a single user on the /r/Cornell subreddit.

Thread title: "{topic_name}"

Thread so far:
{thread_summary}

Your persona:
{persona_snippet}

Now generate exactly ONE Reddit-style comment as this user:
- Casual tone, like a real Cornell student on Reddit
- 1 to 4 sentences
- Include Cornell-specific details when natural (e.g. Ithaca weather, Uris Hall, West campus, Collegetown, course numbers like CS 3110, dorm names, etc.)
- Stay focused on the thread title and what others have said
- Do NOT include any role labels like "User:" or "System:"
- Do NOT use any emojis or emoticons
- Do NOT explain what you are doing, just write the comment itself.
""".strip()



# ---------- Main generation logic ----------

async def generate_topic_thread(
    llm: LLM,
    topic_id: str,
    topic_name: str,
    participants: List[int],
    profiles: Dict[int, Dict[str, Any]],
    starting_msg_id: int,
    n_comments: int,
) -> (Dict[str, Dict[str, Any]], int):
    """
    Generate a single thread (one conversation) for a given topic.

    Returns:
      messages_dict: mapping from message_id (str) to utterance dict
      next_msg_id: next global message id after this topic
    """
    messages: Dict[str, Dict[str, Any]] = {}
    thread_posts: List[Dict[str, Any]] = []
    global_msg_id = starting_msg_id

    for _ in range(n_comments):
        # Choose a random speaker among topic participants
        speaker_agent_id = random.choice(participants)
        profile = profiles[speaker_agent_id]
        speaker_name = profile.get("nickname") or profile.get("name")

        # Build context of the thread so far
        thread_summary = build_thread_summary(thread_posts)
        persona_snippet = build_persona_snippet(profile)
        system_prompt = make_system_prompt(topic_name, thread_summary, persona_snippet)

        # Build dialog for atext_request (system-only is fine here)
        dialog = [
            {"role": "system", "content": system_prompt},
        ]

        # Call AgentSociety LLM: atext_request returns the text directly
        try:
            resp = await llm.atext_request(
                dialog=dialog,
                temperature=0.1,
                max_tokens=256,
            )
        except Exception as e:
            print(f"[WARN] LLM call failed for topic '{topic_name}': {e}")
            continue

        if isinstance(resp, str):
            comment_text = resp.strip()
        else:
            # Fallback in case tools / other structured output is returned
            comment_text = str(resp).strip()

        if not comment_text:
            # Skip empty responses
            continue

        # Choose a parent in this thread to reply to (or None for a new branch)
        if not thread_posts:
            reply_to: Optional[int] = None
        else:
            # 70% chance to reply to a previous comment, 30% to start a "new branch"
            if random.random() < 0.7:
                parent_post = random.choice(thread_posts)
                reply_to = parent_post["id"]
            else:
                reply_to = None

        # Build a Convokit-style utterance
        utterance = {
            "id": global_msg_id,
            "speaker": speaker_name,
            "conversation_id": topic_id,  # thread ID = topic ID from JSON
            "reply_to": reply_to,
            # simple monotonic timestamp; you can scale this or add randomness if needed
            "timestamp": global_msg_id,
            "text": comment_text,
            "meta": {
                "agent_profile_id": speaker_agent_id,
                "topic": topic_name,
            },
        }

        messages[str(global_msg_id)] = utterance
        thread_posts.append(utterance)
        global_msg_id += 1

    return messages, global_msg_id


async def main():
    # 1) Wait for vLLM and load agent profiles + topics
    wait_for_vllm()
    profiles = load_profiles()
    all_agent_ids = list(profiles.keys())
    topics = load_topics()

    # 2) Initialize LLM using your VLLM/Qwen setup
    llm = LLM(
        configs=[
            LLMConfig(
                provider=LLMProviderType.VLLM,
                base_url="http://localhost:8000/v1",
                api_key="sk-123456",  # dummy key or your real one
                model="Qwen/Qwen2.5-7B-Instruct",
                semaphore=200,  # max concurrent requests; tune down if needed
            )
        ]
    )

    # 3) Generate subreddit-style conversations for each topic
    all_messages: Dict[str, Dict[str, Any]] = {}
    global_msg_id = 0

    for topic_id_str, topic_name in topics.items():
        # Sample #participants using your stats (1..53) but capped by actual agents
        max_participants = min(MAX_UNIQUE_USERS, len(all_agent_ids))
        n_participants = sample_truncated_normal_int(
            mean=AVG_UNIQUE_USERS,
            std=2.5,
            lo=1,
            hi=max_participants,
        )
        if len(all_agent_ids) <= n_participants:
            participants = all_agent_ids
        else:
            participants = random.sample(all_agent_ids, n_participants)

        # Sample #comments using your stats (0..127)
        n_comments = sample_truncated_normal_int(
            mean=AVG_COMMENTS,
            std=4.0,
            lo=0,
            hi=MAX_COMMENTS,
        )

        # Skip topics that end up with 0 comments (matches real data where some topics have no comments)
        if n_comments == 0:
            continue

        topic_messages, global_msg_id = await generate_topic_thread(
            llm=llm,
            topic_id=topic_id_str,
            topic_name=topic_name,
            participants=participants,
            profiles=profiles,
            starting_msg_id=global_msg_id,
            n_comments=n_comments,
        )
        all_messages.update(topic_messages)

    # 4) Save to JSON (Convokit-style utterance dict)
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(all_messages, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(all_messages)} subreddit messages to {OUTPUT_PATH}")


if __name__ == "__main__":
    asyncio.run(main())


## Next Steps and Usage Tips

- Once you have generated `cornell_subreddit_conversations_all.json`, you can:
  - Load it back into Python and inspect a few random threads.
  - Convert it into a ConvoKit `Corpus` for further analysis.
  - Use it as a synthetic benchmark for agent-based simulation experiments with AgentSociety.

- You can also tweak:
  - The **distributions** governing number of participants and comments.
  - The **persona fields** and how much they influence the prompts.
  - The **LLM model and parameters** (temperature, max tokens, etc.).

This notebook keeps the original scripts intact so that you can still run them
as standalone modules if you prefer a script-only workflow.
